# 01 - Data Cleaning & Feature Engineering Pipeline
## EV Charging Station Congestion & Grid Load Predictor

This notebook demonstrates the end-to-end data cleaning, anomaly handling, station-hour feature engineering, and academic target proxy derivation from the raw Harvard Dataverse EV charging dataset (`station_data_dataverse.csv`), culminating in the exact 2,000-row project dataset.


### 1. Import Libraries and Inspect Raw Dataset


In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams["figure.figsize"] = (10, 5)

# Locate raw data
raw_path = Path("../data/raw/station_data_dataverse.csv")
df_raw = pd.read_csv(raw_path)
print(f"Raw Dataset Shape: {df_raw.shape}")
df_raw.head()


### 2. Dataset Overview: Types and Null Values


In [ ]:
# Check column data types and missing counts
print("Missing values per column:")
print(df_raw.isnull().sum()[df_raw.isnull().sum() > 0])
print("\nData Types summary:")
print(df_raw.dtypes.value_counts())


### 3. Check Duplicates and Invalid Values


In [ ]:
# Duplicates
dup_count = df_raw.duplicated().sum()
print(f"Duplicate rows: {dup_count}")

# Check negative or zero numerical values
neg_energy = (df_raw['kwhTotal'] <= 0).sum()
neg_duration = (df_raw['chargeTimeHrs'] <= 0).sum()
invalid_hours = ((df_raw['startTime'] < 0) | (df_raw['startTime'] > 23)).sum()

print(f"Zero or negative kWh sessions: {neg_energy}")
print(f"Zero or negative duration sessions: {neg_duration}")
print(f"Invalid startTime hours (<0 or >23): {invalid_hours}")


### 4. Clean Raw Data


In [ ]:
# Essential columns required for analysis
essential_cols = ["stationId", "locationId", "startTime", "weekday", "facilityType", "chargeTimeHrs", "kwhTotal"]

# Drop missing values in essential columns and filter valid physical ranges
df_clean = df_raw.dropna(subset=essential_cols).copy()
df_clean = df_clean[
    (df_clean["kwhTotal"] > 0) &
    (df_clean["chargeTimeHrs"] > 0) &
    (df_clean["startTime"] >= 0) &
    (df_clean["startTime"] <= 23)
].copy()

print(f"Cleaned Usable Records: {len(df_clean)} (from original {len(df_raw)})")


### 5. Feature Engineering: Historical Station-Hour Statistics
Calculate station-hour historical charging behavior:
- `sessions_at_station_hour`: concurrent session activity during that hour
- `avg_energy_kwh`: historical average energy delivered
- `avg_charge_time_min`: historical average charging duration (minutes)


In [ ]:
# Normalize hour and weekday names
weekday_map = {
    "Mon": "Monday", "Tue": "Tuesday", "Wed": "Wednesday",
    "Thu": "Thursday", "Fri": "Friday", "Sat": "Saturday", "Sun": "Sunday"
}
df_clean["hour"] = df_clean["startTime"].astype(int)
df_clean["day_of_week"] = df_clean["weekday"].map(weekday_map).fillna(df_clean["weekday"])

# Calculate historical statistics grouped by stationId and hour
station_hour_stats = df_clean.groupby(["stationId", "hour"]).agg(
    sessions_at_station_hour=("sessionId", "count"),
    avg_energy_kwh=("kwhTotal", "mean"),
    avg_charge_time_min=("chargeTimeHrs", lambda x: x.mean() * 60.0)
).reset_index()

merged = pd.merge(df_clean, station_hour_stats, on=["stationId", "hour"], how="left")
print(f"Merged features shape: {merged.shape}")
merged[["stationId", "hour", "sessions_at_station_hour", "avg_energy_kwh", "avg_charge_time_min"]].head()


### 6. Target Derivation: Congestion Level, Grid Load Level & Waiting Time
- **Congestion Score**: $0.65 \times \text{sessions} + 0.35 \times \text{avg\_duration}$
- **Load Score**: $0.60 \times \text{avg\_energy} + 0.40 \times \text{sessions}$
- Discretized into `Low`, `Medium`, and `High` using 33rd and 66th percentiles.
- **Estimated Waiting Time**: Fraction of average charging time based on congestion tier.


In [ ]:
# 1. Congestion Level
congestion_score = 0.65 * merged["sessions_at_station_hour"] + 0.35 * merged["avg_charge_time_min"]
c_p33 = float(np.percentile(congestion_score, 33))
c_p66 = float(np.percentile(congestion_score, 66))

merged["congestion_level"] = pd.cut(
    congestion_score,
    bins=[-np.inf, c_p33, c_p66, np.inf],
    labels=["Low", "Medium", "High"]
)

# 2. Grid Load Level
load_score = 0.60 * merged["avg_energy_kwh"] + 0.40 * merged["sessions_at_station_hour"]
l_p33 = float(np.percentile(load_score, 33))
l_p66 = float(np.percentile(load_score, 66))

merged["load_level"] = pd.cut(
    load_score,
    bins=[-np.inf, l_p33, l_p66, np.inf],
    labels=["Low", "Medium", "High"]
)

# 3. Estimated Waiting Time
def calc_wait(row):
    c = row["congestion_level"]
    m = row["avg_charge_time_min"]
    if c == "Low":
        w = m * 0.20
    elif c == "Medium":
        w = m * 0.50
    else:
        w = m * 0.85
    return max(0, int(round(w)))

merged["estimated_wait_min"] = merged.apply(calc_wait, axis=1)

print("Congestion Level Distribution:")
print(merged["congestion_level"].value_counts())
print("\nGrid Load Level Distribution:")
print(merged["load_level"].value_counts())


### 7. Select Final Columns and Sample Exactly 2,000 Rows


In [ ]:
# Rename and select final 13 project columns
merged["station_id"] = merged["stationId"]
merged["location_id"] = merged["locationId"]
merged["facility_type"] = merged["facilityType"]
merged["charge_time_hrs"] = merged["chargeTimeHrs"]
merged["energy_kwh"] = merged["kwhTotal"]

final_cols = [
    "station_id", "location_id", "hour", "day_of_week", "facility_type",
    "charge_time_hrs", "energy_kwh", "avg_energy_kwh", "avg_charge_time_min",
    "sessions_at_station_hour", "congestion_level", "load_level", "estimated_wait_min"
]
project_2000 = merged[final_cols].sample(n=2000, random_state=42).reset_index(drop=True)

# Save to processed directory
output_file = Path("../data/processed/ev_charging_project_2000.csv")
output_file.parent.mkdir(parents=True, exist_ok=True)
project_2000.to_csv(output_file, index=False)
print(f"Successfully generated and saved project dataset with shape: {project_2000.shape}")
project_2000.head()


### 8. Data Validation Checklist Verification


In [ ]:
from src.preprocessing import validate_dataset
validate_dataset(output_file)
